In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from preprocess import identity

train = pd.read_pickle('../data/train_proc_title.pkl')
test = pd.read_pickle('../data/test_proc_title.pkl')

vectorizer = TfidfVectorizer(
	preprocessor=identity,
	tokenizer=identity,
	lowercase=False,
	token_pattern=None,
	ngram_range=(1, 2),
	min_df=2
)

X_train = vectorizer.fit_transform(train['lemmas'])
X_test = vectorizer.transform(test['lemmas'])
y_train = train['label']
y_test = test['label']

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

model = LogisticRegression(random_state=67, max_iter=1000)
print("cross_val_score: ", cross_val_score(model, X_train, y_train, cv=10).mean())

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_train_pred = model.predict(X_train)
print("train score: ", accuracy_score(y_train, y_train_pred))


cross_val_score:  0.9608333333333334
train score:  0.983


In [3]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

score = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)
print(score, '\n\n', class_report, '\n\n', matrix)


0.9636666666666667 

               precision    recall  f1-score   support

           0       0.96      0.97      0.96      1500
           1       0.97      0.96      0.96      1500

    accuracy                           0.96      3000
   macro avg       0.96      0.96      0.96      3000
weighted avg       0.96      0.96      0.96      3000
 

 [[1453   47]
 [  62 1438]]


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import pandas as pd

param_grid = {'C': [0.1, 0.3, 1, 3, 10, 30, 100]}

search = GridSearchCV(
	LogisticRegression(max_iter=1000),
	param_grid,
	cv=5,
	scoring='accuracy',
	n_jobs=-1,
	return_train_score=True,
)
search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_)
print(results[['param_C', 'mean_train_score', 'mean_test_score', 'std_test_score']])

print('best C:', search.best_params_['C'])
print('best CV accuracy:', search.best_score_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print('test accuracy:', accuracy_score(y_test, y_pred))

   param_C  mean_train_score  mean_test_score  std_test_score
0      0.1          0.947229         0.937750        0.003532
1      0.3          0.964917         0.950750        0.001889
2      1.0          0.983083         0.959917        0.004236
3      3.0          0.994104         0.965500        0.004137
4     10.0          0.999312         0.966750        0.003021
5     30.0          0.999917         0.967000        0.002974
6    100.0          0.999979         0.967000        0.002601
best C: 30
best CV accuracy: 0.967
test accuracy: 0.968


In [5]:
import joblib
from pathlib import Path

Path('../models').mkdir(exist_ok=True)
joblib.dump(vectorizer, '../models/vectorizer.joblib')
joblib.dump(search.best_estimator_, '../models/model.joblib')

['../models/model.joblib']